# Training dynamics

Figs 5 (end probe) and 8 (mid probe): Δr and r(conf, consistency) along training, at every checkpoint of the development runs. Intermediate checkpoints are evaluated on a fixed 1000-question subsample, the final model (last point) on the full split.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from notebook_helper import *

trained = load_trained(dev=True)   # trained[dataset][model][probe] -> runs, a run = list of checkpoints

In [ ]:
SPLITS = ["train", "test"]
SPLIT_KEYS = {"train": ("train_subset", "train"), "test": ("test",)}   # intermediate / final naming


def checkpoint_corrs(ckpt, split):
    """(r(conf, accuracy), r(conf, consistency)) of one checkpoint, on its own eval questions."""
    entry = next((ckpt[k] for k in SPLIT_KEYS[split] if k in ckpt), None) if ckpt else None
    if entry is None:
        return np.nan, np.nan
    conf = np.asarray(entry["predictions"], dtype=float)
    return nan_pearsonr(entry["accuracies"], conf), nan_pearsonr(entry["consistency"], conf)


def pad_stack(curves):
    """Stack curves of unequal length into a NaN-padded array."""
    curves = [np.asarray(c, dtype=float) for c in curves if len(c)]
    out = np.full((len(curves), max((len(c) for c in curves), default=0)), np.nan)
    for i, c in enumerate(curves):
        out[i, :len(c)] = c
    return out


def dataset_curves(dataset, split, probe):
    """{kind: (epochs, mean, sem)}: average over runs, then mean ± SEM over models."""
    per_model = {"acc": [], "cons": [], "delta": []}
    for m in MODELS:
        runs = {"acc": [], "cons": [], "delta": []}
        for run in trained[dataset][m][probe]:
            r = np.array([checkpoint_corrs(c, split) for c in run], dtype=float)
            runs["acc"].append(r[:, 0])
            runs["cons"].append(r[:, 1])
            runs["delta"].append(r[:, 0] - r[:, 1])
        for kind, curves in runs.items():
            stack = pad_stack(curves)
            if stack.size:
                per_model[kind].append(np.nanmean(stack, axis=0))
    out = {}
    for kind, curves in per_model.items():
        stack = pad_stack(curves)
        if stack.size:
            n_pts = stack.shape[1]              # checkpoints evenly spaced over training
            epochs = EPOCHS_DEV[dataset] * np.arange(1, n_pts + 1) / n_pts
            out[kind] = (epochs, np.nanmean(stack, axis=0), sem(stack, axis=0))
    return out


CURVES = {(d, s, p): dataset_curves(d, s, p)
          for d in DATASETS for s in SPLITS for p in ("end", "mid")}

## Figs 5 and 8 — Δr along training
Colour: split. Mean ± SEM over models; the marker is the final model.

In [ ]:
FS = dict(base=19, title=21, label=21, tick=18, legend=19)
set_style(FS)
SPLIT_COLORS = {"train": COLORS["steel"], "test": COLORS["red"]}
PROBE_STYLES = {"end": dict(ls="-", marker="o"), "mid": dict(ls="--", marker="s")}
DEV_LABELS = {"end": "end probe", "mid": "mid probe"}
ROWS = [("delta", "Δ Pearson r\n(accuracy − consistency)"),
        ("cons", "Pearson r\n(conf, consistency)")]


def plot_development(probes, name):
    fig, axs = plt.subplots(len(ROWS), len(DATASETS), figsize=(17, 8.5),
                            sharey="row", sharex="col", layout="constrained")
    fig.get_layout_engine().set(wspace=0.04, hspace=0.04)
    for i, (kind, ylabel) in enumerate(ROWS):
        row_min = np.inf
        for j, d in enumerate(DATASETS):
            ax = axs[i, j]
            for probe in probes:
                for split in SPLITS:
                    if kind not in CURVES[(d, split, probe)]:
                        continue
                    x, mean, err = CURVES[(d, split, probe)][kind]
                    color = SPLIT_COLORS[split]
                    ax.fill_between(x, mean - err, mean + err, color=color,
                                    alpha=0.18 if len(probes) == 1 else 0.12, lw=0)
                    ax.plot(x, mean, color=color, lw=2.5, ls=PROBE_STYLES[probe]["ls"])
                    ax.plot(x[-1], mean[-1], PROBE_STYLES[probe]["marker"], ms=9, mfc="white",
                            mec=color, mew=2, zorder=4, clip_on=False)   # final model, full split
                    row_min = min(row_min, np.nanmin(mean - err))
            style_ax(ax, lw=0.8)
            ax.margins(x=0.02)
            ax.set_xlim(0, EPOCHS_DEV[d])
            if j:
                ax.tick_params(axis="y", length=0)
            if i == 0:
                ax.set_title(SHORT_LABELS[d], pad=10)
        if row_min < 0:                         # zero line only when the data crosses it
            for ax in axs[i]:
                ax.axhline(0, color="black", lw=1.0, zorder=2)
        axs[i, 0].set_ylabel(ylabel)

    handles = [Line2D([], [], color=SPLIT_COLORS[s], lw=2.5, label=s) for s in SPLITS]
    if len(probes) > 1:
        handles += [Line2D([], [], color="0.3", lw=2.5, ms=8, mfc="white", mew=2,
                           label=DEV_LABELS[p], **PROBE_STYLES[p]) for p in probes]
        fig.legend(handles=handles, loc="outside upper center", ncols=len(handles),
                   frameon=False, handlelength=2.4, columnspacing=1.8)
        fig.supxlabel(f"epochs   ·   mean ± SEM over {len(MODELS)} models", fontsize=FS["label"])
    else:
        axs[0, 0].legend(handles=handles, loc="upper left", frameon=False, handlelength=1.6)
        fig.supxlabel("epochs", fontsize=FS["label"])
        fig.suptitle(f"{DEV_LABELS[probes[0]]}   ·   mean ± SEM over {len(MODELS)} models",
                     fontsize=FS["base"])
    save_fig(fig, name)
    plt.show()

In [ ]:
plot_development(["end"], "fig05_development_end")

In [ ]:
plot_development(["mid"], "fig08_development_mid")